In [0]:
%run ./00_Organizacao_do_Ambiente

bronze_schema: workspace.bronze
silver_schema: workspace.silver
gold_schema: workspace.gold
landing_path: /Volumes/workspace/rocket/cinedata_raw


In [0]:
# 1. Receita total em R$
display(spark.sql("SELECT SUM(receita_brl) AS receita_total_brl FROM gold.fact_movies_performance"))

# 2. Top 5 filmes por popularidade
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON f.sk_movie_id = m.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

# 3. Quantidade de filmes por gênero
display(spark.sql("""
    SELECT g.nome_genero, COUNT(*) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON b.sk_genre_id = g.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))

# 4. Top 10 filmes por receita (USD e BRL) com RANK()
display(spark.sql("""
    SELECT titulo, receita_usd, receita_brl, ranking FROM (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS ranking
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON f.sk_movie_id = m.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    ) WHERE ranking <= 10
    ORDER BY ranking
"""))

# 5. Ator com mais participações nos últimos 2 anos
display(spark.sql("""
    WITH data_limite AS (
        SELECT MAX(data_lancamento) AS max_data FROM gold.dim_movies WHERE data_lancamento <= current_date()
    )
    SELECT p.nome_pessoa, COUNT(*) AS qtd_participacoes
    FROM gold.bridge_movie_person bp
    JOIN gold.dim_people p ON bp.sk_person_id = p.sk_person_id
    JOIN gold.dim_movies m ON bp.sk_movie_id = m.sk_movie_id
    CROSS JOIN data_limite dl
    WHERE p.tipo_pessoa = 'Ator'
      AND m.data_lancamento > add_months(dl.max_data, -24)
      AND m.data_lancamento <= dl.max_data
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC
    LIMIT 1
"""))

# 6. Produtora com maior lucro nos últimos 5 anos
display(spark.sql("""
    WITH data_limite AS (
        SELECT MAX(data_lancamento) AS max_data FROM gold.dim_movies WHERE data_lancamento <= current_date()
    )
    SELECT c.nome_produtora, SUM(f.lucro_usd) AS lucro_total_usd
    FROM gold.bridge_movie_company bc
    JOIN gold.dim_companies c ON bc.sk_company_id = c.sk_company_id
    JOIN gold.dim_movies m ON bc.sk_movie_id = m.sk_movie_id
    JOIN gold.fact_movies_performance f ON f.sk_movie_id = m.sk_movie_id
    CROSS JOIN data_limite dl
    WHERE m.data_lancamento > add_months(dl.max_data, -60)
      AND m.data_lancamento <= dl.max_data
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_usd DESC
    LIMIT 1
"""))

receita_total_brl
827318778947.13


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


nome_genero,qtd_filmes
Drama,32287
Documentary,18996
Comedy,18625
Thriller,10276
Horror,9729
Romance,7639
Action,6049
Crime,4747
Animation,4470
TV Movie,4080


titulo,receita_usd,receita_brl,ranking
Avengers: Endgame,2800000000.00,14311080000.00,1
Avatar: The Way of Water,2320250281.00,11859031211.22,2
AVENGERS: INFINITY WAR,2052415039.00,10490098505.83,3
spider-man: no way home,1921847111.00,9822752769.03,4
The Lion King,1663075401.00,8500144682.05,5
Top Gun: Maverick,1488732821.00,7609062321.41,6
Barbie,1428545028.00,7301436492.61,7
The Super Mario Bros. Movie,1355725263.00,6929247391.72,8
Black Panther,1349926083.00,6899607202.82,9
Star Wars: The Last Jedi,1332698830.00,6811556990.01,10


nome_pessoa,qtd_participacoes
Kevin Hart,66


nome_produtora,lucro_total_usd
Universal Pictures,5772329679.00
